In [1]:
!git clone https://github.com/d12eek/Kidney-Disease-Classification-Deep-Learning-Project.git
%cd Kidney-Disease-Classification-Deep-Learning-Project
!pip install mlflow ensure -q
!pip install -e . -q
print("Done!")

Cloning into 'Kidney-Disease-Classification-Deep-Learning-Project'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 95 (delta 35), reused 87 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 154.68 KiB | 386.00 KiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/Kidney-Disease-Classification-Deep-Learning-Project
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 133.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.5 MB/s eta 0:00:00
 

In [2]:
import os
os.chdir("/content/Kidney-Disease-Classification-Deep-Learning-Project")

!mkdir -p ~/.kaggle
!echo '{"username":"d12eek","key":"KGAT_26d8846b85b6a8fba229bc4ed1a8c2c8"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone
!unzip -q ct-kidney-dataset-normal-cyst-tumor-and-stone.zip -d temp_data

import shutil
base = "temp_data/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
os.makedirs("artifacts/data_ingestion/kidney-ct-scan-image", exist_ok=True)
shutil.copytree(f"{base}/Normal", "artifacts/data_ingestion/kidney-ct-scan-image/Normal", dirs_exist_ok=True)
shutil.copytree(f"{base}/Tumor", "artifacts/data_ingestion/kidney-ct-scan-image/Tumor", dirs_exist_ok=True)
shutil.rmtree("temp_data")
!rm ct-kidney-dataset-normal-cyst-tumor-and-stone.zip

normal = len(os.listdir("artifacts/data_ingestion/kidney-ct-scan-image/Normal"))
tumor = len(os.listdir("artifacts/data_ingestion/kidney-ct-scan-image/Tumor"))
print(f"Normal: {normal}, Tumor: {tumor} — Dataset ready!")

Dataset URL: https://www.kaggle.com/datasets/nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone
License(s): Attribution 4.0 International (CC BY 4.0)
100% 1.52G/1.52G [01:15<00:00, 21.6MB/s]

Normal: 5077, Tumor: 2283 — Dataset ready!


In [3]:
import os
os.chdir("/content/Kidney-Disease-Classification-Deep-Learning-Project")
import tensorflow as tf

# Generators
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.20,
    rotation_range=40,
    horizontal_flip=True,
    zoom_range=0.2
)

train_data = train_gen.flow_from_directory(
    "artifacts/data_ingestion/kidney-ct-scan-image",
    target_size=(224, 224),
    batch_size=16,
    subset="training",
    shuffle=True
)

val_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.20
)

val_data = val_gen.flow_from_directory(
    "artifacts/data_ingestion/kidney-ct-scan-image",
    target_size=(224, 224),
    batch_size=16,
    subset="validation",
    shuffle=False
)

# VGG16 architecture build karo
base_model = tf.keras.applications.VGG16(
    input_shape=(224, 224, 3),
    weights='imagenet',
    include_top=False
)
base_model.trainable = False
flatten = tf.keras.layers.Flatten()(base_model.output)
output = tf.keras.layers.Dense(2, activation='softmax')(flatten)
model = tf.keras.Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Class weights
class_weight = {
    0: (5077 + 2283) / (2 * 5077),  # Normal
    1: (5077 + 2283) / (2 * 2283)   # Tumor
}
print("Class weights:", class_weight)

# Train
model.fit(
    train_data,
    epochs=15,
    validation_data=val_data,
    class_weight=class_weight
)

# Save weights
os.makedirs("artifacts/training", exist_ok=True)
model.save_weights("artifacts/training/model_weights.h5")
print("Done!")

Found 5889 images belonging to 2 classes.
Found 1471 images belonging to 2 classes.
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Class weights: {0: 0.7248375024620839, 1: 1.6119141480508103}
Epoch 1/15
369/369 ━━━━━━━━━━━━━━━━━━━━ 129s 327ms/step - accuracy: 0.6137 - loss: 0.6602 - val_accuracy: 0.5697 - val_loss: 0.7126
Epoch 2/15
369/369 ━━━━━━━━━━━━━━━━━━━━ 107s 289ms/step - accuracy: 0.7123 - loss: 0.5687 - val_accuracy: 0.6941 - val_loss: 0.6403
Epoch 3/15
369/369 ━━━━━━━━━━━━━━━━━━━━ 107s 290ms/step - accuracy: 0.7759 - loss: 0.5144 - val_accuracy: 0.5894 - val_loss: 0.6776
Epoch 4/15
369/369 ━━━━━━━━━━━━━━━━━━━━ 106s 287ms/step - accuracy: 0.8078 - loss: 0.4722 - val_accuracy: 0.6798 - val_loss: 0.6089
Epoch 5/15
369/369 ━━━━━━━━━━━━━━━━━━━━ 107s 289ms/step - accuracy: 0.8244 - loss: 0.4456 - val_accuracy: 0.7294 - val_loss: 0.5653
Epoch 6/15
369/369 ━━━━━━━━━━━━━━━━━━━━ 107s 290ms/step - accuracy: 0.8365 - loss: 0.4232 - val_accuracy: 0.7111 - val_loss: 0.5643
Epoch 7/15


ValueError: The filename must end in `.weights.h5`. Received: filepath=artifacts/training/model_weights.h5

In [4]:
import os
os.chdir("/content/Kidney-Disease-Classification-Deep-Learning-Project")

os.makedirs("artifacts/training", exist_ok=True)
model.save_weights("artifacts/training/model_weights.weights.h5")
print("Done!")

from google.colab import files
files.download("artifacts/training/model_weights.weights.h5")

Done!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import os
os.chdir("/content/Kidney-Disease-Classification-Deep-Learning-Project")

# SavedModel format mein save karo
model.export("artifacts/training/saved_model")

# Zip karke download karo
import shutil
shutil.make_archive("saved_model", 'zip', "artifacts/training/saved_model")

from google.colab import files
files.download("saved_model.zip")
print("Done!")

Saved artifact at 'artifacts/training/saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  139558048278352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139558048279504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139558048280464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139558048279312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139558048279120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139558048281040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139558048281616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139558048282384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139558048283344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139558048281232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!
